In [0]:

from pyspark.sql.types import *
import pyspark.sql.functions as F
import pyspark.sql.functions  as F
from pyspark.sql.functions import concat_ws
from common_utils.scd_functions import scd1
from common_utils.logging import get_logger

logger = get_logger("sales_silver")

logger.info("Reading sales data from Bronze")
df = spark.read.table("retaildataplatform.bronze.s3_sales")
logger.info("Bronze sales data successfully loaded")

def transform_sales(df):

    product_schema = StructType([
        StructField("curr", StringType(), True),
        StructField("id", StringType(), True),
        StructField("name", StringType(), True),
        StructField("price", DoubleType(), True),
        StructField("qty", IntegerType(), True),
        StructField("unit", StringType(), True)
    ])

    df = (df.withColumn("product",F.from_json(F.col("product"), product_schema)))

    return df.select(
        "customer_id",
        "customer_name",
        "product_name",
        "order_date",
        "product_category",
        F.col("product.curr").alias("currency"),
        F.col("product.id").alias("product_id"),
        F.col("product.name").alias("product_detail_name"),
        F.col("product.price").alias("price"),
        F.col("product.qty").alias("product_qty"),
        F.col("product.unit").alias("product_unit"),
        "total_price",
        "last_update_ts"
    )
    

cleaned_sales = transform_sales(df)
cleaned_sales = cleaned_sales.filter(F.col("product_id").isNotNull()).dropDuplicates()

logger.info("Sales transformation completed")
cleaned_sales.display()

scd1(
    spark,
    cleaned_sales,
    [
        "customer_id",
        "order_date",
        "product_id",
        "price",
        "product_qty",
        "total_price"
    ],
    "retaildataplatform.silver.sales"
)





In [0]:
cleaned_sales \
    .groupBy(
        "customer_id",
        "order_date",
        "product_id",
        "price",
        "product_qty",
        "total_price"
    ) \
    .count() \
    .filter(F.col("count") > 1) \
    .show()